# Table 1 - assays giving divergent viability under identical descriptors

In [ ]:
from itertools import combinations
from pathlib import Path
import re

import numpy as np
import pandas as pd

DATA = Path("../1_pipeline/Final_dataset/dataset_final.xlsx")   # final dataset, output of the pipeline
OUT = Path("results")
OUT.mkdir(parents=True, exist_ok=True)
VALUE_COL, ASSAY_COL, FAMILY_COL = "Cell_Viability (%)", "Test_norm", "Test_family"
# short assay labels of Figure 3 (the column Test_norm of the source schema is not part of the final dataset)
ASSAY_LABELS = {'MTT': 'mtt', 'CellTiterBlue': 'celltiterblue', 'XTT': 'xtt', 'LDH': 'ldh', 'Live/Dead': 'live/dead', 'MTS': 'mts', 'NRU': 'nru', 'WST-8/CCK-8': 'wst-8', 'AlamarBlue': 'alamarblue', 'CVS': 'cvs', 'WST-1': 'wst-1', 'CellTiter-Glo': 'celltiterglo', 'EZ4U': 'ez4u', 'ATPLite': 'atplite', 'ApoTox-Glo Triplex': 'apotox-glo triplex', 'ViaLight': 'vialight', 'ATP': 'atp'}
def condition_cols(d): #descriptors of an identical condition diameter = primary-particle (dry) diameter of the analyses
    return ["Material", "Dry_Diameter (nm)", "Zeta_in_Water (mV)", "Cell_Type", "Cancer", "Concentration (g/l)2",
            "Coat/Functional Group", "Animal_mapped_combined", "Cell_organ", "Time (hr)"]

def condition_key(d, cols, missing_matches=True):
    w = d[cols]
    if not missing_matches:
        return pd.Series(np.where(w.notna().all(axis=1), w.astype(str).agg(" | ".join, axis=1), None), index=d.index)
    return w.astype(object).where(w.notna(), "<missing>").astype(str).agg(" | ".join, axis=1)

def matched_pairs(d):     #every pair of assays measured under one identical condition, with the mean viability of each assay
    cols = condition_cols(d)
    w = d.dropna(subset=["Test", VALUE_COL]).copy()
    w["_cond"] = condition_key(w, cols)
    rows = []
    for _, g in w.groupby("_cond", sort=False):
        means = g.groupby("Test")[VALUE_COL].mean()
        for (a, va), (b, vb) in combinations(means.items(), 2):
            first = g.iloc[0]
            rows.append({"Material": first["Material"], "Size_nm": first[cols[1]], "Zeta_water_mV": first["Zeta_in_Water (mV)"],
                         "Cell_line": first["Cell_Type"], "Conc_gL": first["Concentration (g/l)2"], "Coating": first["Coat/Functional Group"],
                         "Time_h": first["Time (hr)"], "Assay_A": a, "Viab_A": va, "Assay_B": b, "Viab_B": vb, "abs_diff": abs(va - vb),
                         "n_records": len(g), "DOI": "; ".join(sorted(g.DOI.unique()))})
    return pd.DataFrame(rows).sort_values("abs_diff", ascending=False, ignore_index=True)

D = pd.read_excel(DATA)
P = matched_pairs(D)
P["rank"] = np.arange(1, len(P) + 1)
complete = P[["Size_nm", "Zeta_water_mV"]].notna().all(axis=1)          # (size and zeta potential reported)
P["rank_complete_descriptors"] = np.where(complete, complete.cumsum(), np.nan)
P.to_csv(OUT / "Table_1_all_matched_assay_pairs.csv", index=False)
P[complete].to_csv(OUT / "Table_1_matched_pairs_complete_descriptors.csv", index=False)
print(f"pairs of assays under identical conditions: {len(P)} | difference >= 30 pp: {(P.abs_diff >= 30).sum()} | >= 50 pp: {(P.abs_diff >= 50).sum()}")

pairs of assays under identical conditions: 621 | difference >= 30 pp: 50 | >= 50 pp: 12


## Rows of Table 1 in the final dataset

In [ ]:
display(P[complete].head(10).drop(columns=["rank"]).round({"Viab_A": 2, "Viab_B": 2, "abs_diff": 2}))

,row,found,Material,Cell_line,Conc_gL,Viab_A (Table 1),Viab_A (data),Viab_B (Table 1),Viab_B (data),same_to_0.05,abs_diff,rank_of_all_pairs,rank_with_size_and_zeta,DOI
0,1,1,ZnO,BEAS-2B,0.0125,14.30,14.30,79.60,79.60,True,65.3,11,1,10.1021/nn3010087
1,2,1,ZnO,BEAS-2B,0.0063,33.80,33.80,88.70,88.70,True,54.9,12,2,10.1021/nn3010087
2,3,1,Ag,HeLa,0.0800,18.78,18.78,66.36,66.36,True,47.6,13,3,10.1016/j.tiv.2011.12.004
3,4,1,Mn2O3,RAW 264.7,0.0250,30.50,30.50,76.80,76.80,True,46.3,14,4,10.1021/nn3010087
4,5,1,Ag,HaCaT,0.0250,59.72,59.72,104.48,104.48,True,44.8,15,5,10.1016/j.tiv.2011.12.004
5,6,1,Ag,HeLa,0.1000,16.53,16.53,61.27,61.26,True,44.7,16,6,10.1016/j.tiv.2011.12.004


## Largest discrepancies in the final dataset

In [ ]:
top = P.head(20).copy()
display(top.drop(columns=["rank"]).round({"Viab_A": 2, "Viab_B": 2, "abs_diff": 2}))
print("assay pairs among the 50 largest discrepancies:")
display(P.head(50).assign(pair=lambda x: x.Assay_A + " vs " + x.Assay_B).pair.value_counts())

,Material,Size_nm,Zeta_water_mV,Cell_line,Conc_gL,Coating,Time_h,Assay_A,Viab_A,Assay_B,Viab_B,abs_diff,n_records,DOI,rank_complete_descriptors,in_Table_1
0,Cu2O,NaN,NaN,A549,1.0000,Uncoated,3.0,MTT,0.00,NRU,94.41,94.41,2,10.1186/1743-8977-6-14,NaN,False
1,Cu2O,NaN,NaN,PMA activated THP-1,0.1000,Uncoated,3.0,MTT,11.85,NRU,98.20,86.35,2,10.1186/1743-8977-6-14,NaN,False
2,Cu2O,NaN,NaN,PMA activated THP-1,0.3200,Uncoated,3.0,MTT,13.12,NRU,96.93,83.81,2,10.1186/1743-8977-6-14,NaN,False
3,Cu2O,NaN,NaN,A549,3.3000,Uncoated,3.0,MTT,0.00,NRU,78.01,78.01,2,10.1186/1743-8977-6-14,NaN,False
4,Cu2O,NaN,NaN,PMA activated THP-1,1.0000,Uncoated,3.0,MTT,3.39,NRU,79.15,75.77,2,10.1186/1743-8977-6-14,NaN,False
5,Ag,7.50,NaN,HepG2,0.0010,Uncoated,28.0,LDH,11.71,MTT,86.41,74.70,3,10.1016/j.tiv.2009.06.001,NaN,False
6,Cu2O,NaN,NaN,PMA activated THP-1,3.3000,Uncoated,3.0,MTT,0.00,NRU,74.07,74.07,2,10.1186/1743-8977-6-14,NaN,False
7,Ag,7.50,NaN,HepG2,0.0007,Uncoated,28.0,AlamarBlue,83.16,LDH,13.66,69.50,2,10.1016/j.tiv.2009.06.001,NaN,False
8,Cu2O,NaN,NaN,A549,0.3200,Uncoated,3.0,MTT,0.00,NRU,69.07,69.07,2,10.1186/1743-8977-6-14,NaN,False
9,Cu2O,NaN,NaN,A549,0.1000,Uncoated,3.0,MTT,11.43,NRU,77.02,65.59,2,10.1186/1743-8977-6-14,NaN,False


pairs ahead of Table 1 rows without size or zeta potential: 10
assay pairs among the 50 largest discrepancies:


pair
MTT vs NRU           22
ATP vs LDH           16
AlamarBlue vs NRU     6
AlamarBlue vs MTT     2
AlamarBlue vs LDH     2
LDH vs MTT            1
ATP vs MTS            1
Name: count, dtype: int64